In [6]:
import os, json, math, random, string, zipfile
import numpy as np
from collections import Counter
from dataclasses import dataclass
from typing import List, Dict, Tuple, Any
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from huggingface_hub import hf_hub_download, login

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"

HF_DATASET_REPO = "Exploration-Lab/CS-781-Capstone"
try:
    from kaggle_secrets import UserSecretsClient
    HF_TOKEN = UserSecretsClient().get_secret("HF_TOKEN")
except Exception:
    HF_TOKEN = ""
if HF_TOKEN:
    login(token=HF_TOKEN)

TOKENIZER_PATH = "/kaggle/input/datasets/brindaaltaccount/legal-bpe-tokenizer/custom_bpe.json"
CHECKPOINT_PATH = "/kaggle/input/datasets/brindafadadu/cs781-legal-model-checkpoints/decoder_lm.pt"

MAX_SEQ_LEN = 512
STRIDE = 128
BATCH_SIZE = 4
GRAD_ACCUM_STEPS = 4
EPOCHS = 6
BACKBONE_LR = 5e-5
HEAD_LR = 1e-3
USE_VAL = True
CAUSAL = False

LABEL_NAMES = ["APP", "RESP", "A.COUNSEL", "R.COUNSEL", "JUDGE",
               "WIT", "AUTH", "COURT", "STAT", "PREC", "DATE", "CASENO"]
ID2LABEL = {0: "O"}
LABEL2ID = {"O": 0}
for idx, lbl in enumerate(LABEL_NAMES):
    ID2LABEL[1 + 2 * idx] = f"B-{lbl}"
    LABEL2ID[f"B-{lbl}"] = 1 + 2 * idx
    ID2LABEL[2 + 2 * idx] = f"I-{lbl}"
    LABEL2ID[f"I-{lbl}"] = 2 + 2 * idx
NUM_CLASSES = len(ID2LABEL)
IGNORE_INDEX = -100
print(f"Device: {DEVICE} | Classes: {NUM_CLASSES} | USE_VAL={USE_VAL} | CAUSAL={CAUSAL}")

Device: cuda | Classes: 25 | USE_VAL=True | CAUSAL=False


In [7]:
SPECIALS = ["<pad>", "<unk>", "<s>", "</s>"]
END_OF_WORD = "</w>"

class ByteBPETokenizer:
    def __init__(self, vocab_size=8000):
        self.vocab_size = vocab_size
        self.merges = []
        self.merge_rank = {}
        self.itos = []
        self.stoi = {}

    @classmethod
    def load(cls, path):
        with open(path, "r", encoding="utf-8") as f:
            data = json.load(f)
        tok = cls(vocab_size=data["vocab_size"])
        tok.itos = data["itos"]
        tok.stoi = {t: i for i, t in enumerate(tok.itos)}
        tok.merges = [tuple(p) for p in data["merges"]]
        tok.merge_rank = {p: i for i, p in enumerate(tok.merges)}
        return tok

    def _word_to_symbols(self, word):
        return list(word.encode("utf-8").decode("latin-1")) + [END_OF_WORD]

    def _apply_merges_to_word(self, word):
        symbols = self._word_to_symbols(word)
        while len(symbols) > 1:
            pairs_here = [(symbols[i], symbols[i + 1]) for i in range(len(symbols) - 1)]
            candidates = [(self.merge_rank[p], idx) for idx, p in enumerate(pairs_here) if p in self.merge_rank]
            if not candidates:
                break
            _, idx = min(candidates)
            merged = symbols[idx] + symbols[idx + 1]
            symbols = symbols[:idx] + [merged] + symbols[idx + 2:]
        return symbols

    def encode_with_exact_offsets(self, text):
        tokens, offsets = [], []
        i, n = 0, len(text)
        while i < n:
            while i < n and text[i].isspace():
                i += 1
            if i >= n:
                break
            word_start = i
            while i < n and not text[i].isspace():
                i += 1
            word_end = i
            subwords = self._apply_merges_to_word(text[word_start:word_end])
            curr_pos = word_start
            for sym in subwords:
                clean_sym = sym.replace(END_OF_WORD, "")
                try:
                    sym_chars = clean_sym.encode("latin-1").decode("utf-8", errors="replace")
                except Exception:
                    sym_chars = clean_sym
                sub_start = curr_pos
                sub_end = min(word_end, curr_pos + len(sym_chars))
                curr_pos = sub_end
                tokens.append(self.stoi.get(sym, self.stoi.get("<unk>", 1)))
                offsets.append((sub_start, sub_end))
        return tokens, offsets

tokenizer = ByteBPETokenizer.load(TOKENIZER_PATH)
print("Tokenizer vocab size:", tokenizer.vocab_size)

Tokenizer vocab size: 8000


In [8]:
def download_phase2_file(filename: str) -> str:
    return hf_hub_download(
        repo_id=HF_DATASET_REPO,
        filename=filename,
        repo_type="dataset",
        token=HF_TOKEN
    )

print("Downloading dataset files...")
f1_text_path = download_phase2_file("Phase_2_LNER/Train/fold1-text.json")
f1_gold_path = download_phase2_file("Phase_2_LNER/Train/fold1-gold.json")
f2_text_path = download_phase2_file("Phase_2_LNER/Train/fold2-text.json")
f2_gold_path = download_phase2_file("Phase_2_LNER/Train/fold2-gold.json")
f3_text_path = download_phase2_file("Phase_2_LNER/Test/fold3-text.json")

with open(f1_text_path, "r", encoding="utf-8") as f: train_texts = json.load(f)
with open(f1_gold_path, "r", encoding="utf-8") as f: train_golds = json.load(f)
with open(f2_text_path, "r", encoding="utf-8") as f: val_texts = json.load(f)
with open(f2_gold_path, "r", encoding="utf-8") as f: val_golds = json.load(f)
with open(f3_text_path, "r", encoding="utf-8") as f: test_texts = json.load(f)

# ---- NEW: held-out validation switch ----
if USE_VAL:
    tr_texts, tr_golds = train_texts, train_golds
else:
    tr_texts = {**train_texts, **val_texts}
    tr_golds = {**train_golds, **val_golds}
print(f"Training docs: {len(tr_texts)} | USE_VAL={USE_VAL}")

# ---- NEW: windows where the last one is always full-length (no tiny padded tail) ----
def window_starts(n, max_len=MAX_SEQ_LEN, stride=STRIDE):
    if n <= max_len:
        return [0]
    starts = list(range(0, n - max_len + 1, max_len - stride))
    if starts[-1] + max_len < n:
        starts.append(n - max_len)
    return starts

class ExactLNERDataset(Dataset):
    def __init__(self, docs: Dict[str, str], golds: Dict[str, List[Any]], tokenizer: ByteBPETokenizer,
                 max_len: int = MAX_SEQ_LEN, stride: int = STRIDE):
        self.samples = []
        for doc_id, text in docs.items():
            token_ids, offsets = tokenizer.encode_with_exact_offsets(text)

            token_labels = [0] * len(token_ids)
            if doc_id in golds:
                for annot in golds[doc_id]:
                    g_start, g_end, g_lbl = annot[0], annot[1], annot[2]
                    first_matched = False
                    for idx, (t_s, t_e) in enumerate(offsets):
                        if max(t_s, g_start) < min(t_e, g_end):
                            if not first_matched:
                                token_labels[idx] = LABEL2ID.get(f"B-{g_lbl}", 0)
                                first_matched = True
                            else:
                                token_labels[idx] = LABEL2ID.get(f"I-{g_lbl}", 0)

            for start_idx in window_starts(len(token_ids), max_len, stride):
                end_idx = min(len(token_ids), start_idx + max_len)
                c_ids = token_ids[start_idx:end_idx]
                c_lbls = token_labels[start_idx:end_idx]

                pad_len = max_len - len(c_ids)
                padded_ids = c_ids + [0] * pad_len
                padded_lbls = c_lbls + [IGNORE_INDEX] * pad_len

                self.samples.append({
                    "input_ids": torch.tensor(padded_ids, dtype=torch.long),
                    "labels": torch.tensor(padded_lbls, dtype=torch.long)
                })

    def __len__(self):
        return len(self.samples)

    def __getitem__(self, idx):
        return self.samples[idx]

train_ds = ExactLNERDataset(tr_texts, tr_golds, tokenizer)
train_loader = DataLoader(train_ds, batch_size=BATCH_SIZE, shuffle=True)
print(f"Total training sliding windows: {len(train_ds)}")

fold1-text.json:   0%|          | 0.00/1.11M [00:00<?, ?B/s]

fold1-gold.json:   0%|          | 0.00/461k [00:00<?, ?B/s]

fold2-text.json:   0%|          | 0.00/1.12M [00:00<?, ?B/s]

fold2-gold.json:   0%|          | 0.00/475k [00:00<?, ?B/s]

fold3-text.json:   0%|          | 0.00/1.19M [00:00<?, ?B/s]

Training docs: 35 | USE_VAL=True
Total training sliding windows: 652


In [9]:
@dataclass
class LMConfig:
    vocab_size: int
    d_model: int = 768
    n_layers: int = 12
    n_heads: int = 12
    n_kv_heads: int = 12
    d_ff: int = 3072
    max_seq_len: int = 1024
    dropout: float = 0.1
    pos_emb_type: str = "sinusoidal"
    attention_scale: float = None
    norm_first: bool = True

def sinusoidal_positional_encoding(max_len: int, d_model: int) -> torch.Tensor:
    pe = torch.zeros(max_len, d_model)
    position = torch.arange(0, max_len, dtype=torch.float).unsqueeze(1)
    i = torch.arange(0, d_model // 2, dtype=torch.float)
    div_term = torch.pow(10000.0, 2 * i / d_model)
    pe[:, 0::2] = torch.sin(position / div_term)
    pe[:, 1::2] = torch.cos(position / div_term)
    return pe

class CausalSelfAttention(nn.Module):
    def __init__(self, d_model, n_heads, dropout=0.0, attention_scale=None):
        super().__init__()
        assert d_model % n_heads == 0
        self.n_heads = n_heads
        self.d_head = d_model // n_heads
        self.attention_scale = (1.0 / math.sqrt(self.d_head)) if attention_scale is None else float(attention_scale)
        self.q_proj = nn.Linear(d_model, d_model, bias=False)
        self.k_proj = nn.Linear(d_model, d_model, bias=False)
        self.v_proj = nn.Linear(d_model, d_model, bias=False)
        self.out_proj = nn.Linear(d_model, d_model, bias=False)
        self.attn_dropout = nn.Dropout(dropout)
        self.resid_dropout = nn.Dropout(dropout)

    def forward(self, x):
        B, T, C = x.shape
        q = self.q_proj(x).view(B, T, self.n_heads, self.d_head).transpose(1, 2)
        k = self.k_proj(x).view(B, T, self.n_heads, self.d_head).transpose(1, 2)
        v = self.v_proj(x).view(B, T, self.n_heads, self.d_head).transpose(1, 2)
        z = F.scaled_dot_product_attention(
            q, k, v,
            dropout_p=self.attn_dropout.p if self.training else 0.0,
            is_causal=CAUSAL,
            scale=self.attention_scale,
        )
        z = z.transpose(1, 2).contiguous().view(B, T, C)
        return self.resid_dropout(self.out_proj(z))

class FeedForward(nn.Module):
    def __init__(self, d_model, d_ff, dropout):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(d_model, d_ff),
            nn.GELU(),
            nn.Dropout(dropout),
            nn.Linear(d_ff, d_model),
            nn.Dropout(dropout),
        )

    def forward(self, x):
        return self.net(x)

class Block(nn.Module):
    def __init__(self, cfg):
        super().__init__()
        self.norm_first = cfg.norm_first
        self.ln1 = nn.LayerNorm(cfg.d_model)
        self.attn = CausalSelfAttention(cfg.d_model, cfg.n_heads, cfg.dropout, cfg.attention_scale)
        self.ln2 = nn.LayerNorm(cfg.d_model)
        self.ffn = FeedForward(cfg.d_model, cfg.d_ff, cfg.dropout)

    def forward(self, x):
        if self.norm_first:
            x = x + self.attn(self.ln1(x))
            x = x + self.ffn(self.ln2(x))
        else:
            x = self.ln1(x + self.attn(x))
            x = self.ln2(x + self.ffn(x))
        return x

class Arch1NERModel(nn.Module):
    def __init__(self, cfg, num_classes=NUM_CLASSES):
        super().__init__()
        self.cfg = cfg
        self.tok_emb = nn.Embedding(cfg.vocab_size, cfg.d_model)
        self.register_buffer("pos_emb", sinusoidal_positional_encoding(cfg.max_seq_len, cfg.d_model), persistent=False)
        self.blocks = nn.ModuleList([Block(cfg) for _ in range(cfg.n_layers)])
        self.ln_f = nn.LayerNorm(cfg.d_model)
        self.classifier = nn.Sequential(
            nn.Linear(cfg.d_model, cfg.d_model),
            nn.LayerNorm(cfg.d_model),
            nn.GELU(),
            nn.Dropout(cfg.dropout),
            nn.Linear(cfg.d_model, num_classes),
        )

    def forward(self, idx):
        B, T = idx.shape
        x = self.tok_emb(idx) + self.pos_emb[:T, :]
        for block in self.blocks:
            x = block(x)
        return self.classifier(self.ln_f(x))

ckpt = torch.load(CHECKPOINT_PATH, map_location="cpu", weights_only=False)
state_dict = ckpt["model_state_dict"] if "model_state_dict" in ckpt else ckpt
cfg = ckpt.get("config") if isinstance(ckpt, dict) else None
if cfg is None:
    cfg = LMConfig(
        vocab_size=state_dict["tok_emb.weight"].shape[0],
        d_model=state_dict["tok_emb.weight"].shape[1],
        n_layers=max(int(k.split(".")[1]) for k in state_dict if k.startswith("blocks.")) + 1,
        d_ff=state_dict["blocks.0.ffn.net.0.weight"].shape[0],
    )
assert cfg.pos_emb_type == "sinusoidal", "This model class only supports sinusoidal positions"
assert "blocks.0.ln1.weight" in state_dict and "blocks.0.attn.q_proj.weight" in state_dict, "Not an Arch-1 checkpoint"
assert tokenizer.vocab_size == state_dict["tok_emb.weight"].shape[0], "Tokenizer and checkpoint vocab sizes differ"

model = Arch1NERModel(cfg).to(DEVICE)
backbone_dict = {k: v for k, v in state_dict.items() if not k.startswith("head")}
missing, unexpected = model.load_state_dict(backbone_dict, strict=False)
print("MISSING   :", len(missing), missing[:8])
print("UNEXPECTED:", len(unexpected), unexpected[:8])
assert all(m.startswith("classifier.") for m in missing) and not unexpected, "Backbone weights did NOT load"

for p in model.parameters():
    p.requires_grad = True
n_params = sum(p.numel() for p in model.parameters())
print(f"Arch-1 full fine-tuning | parameters: {n_params:,} ({n_params / 1e6:.1f}M) | heads={cfg.n_heads} layers={cfg.n_layers} d_model={cfg.d_model}")

MISSING   : 6 ['classifier.0.weight', 'classifier.0.bias', 'classifier.1.weight', 'classifier.1.bias', 'classifier.4.weight', 'classifier.4.bias']
UNEXPECTED: 0 []
Arch-1 full fine-tuning | parameters: 91,774,489 (91.8M) | heads=12 layers=12 d_model=768


In [10]:
STRIP_CHARS = " \t\n\r" + string.punctuation + "\u2018\u2019\u201c\u201d\u2013\u2014\u2022\u00b7\u2026"

def window_starts(n, max_len=MAX_SEQ_LEN, stride=STRIDE):
    if n <= max_len:
        return [0]
    starts = list(range(0, n - max_len + 1, max_len - stride))
    if starts[-1] + max_len < n:
        starts.append(n - max_len)
    return starts

def clean_span(text, start, end, label):
    while start < end and text[start] in STRIP_CHARS:
        start += 1
    while end > start and text[end - 1] in STRIP_CHARS:
        end -= 1
    return start, end, label

def decode_spans_exact(doc_text, pred_ids, offsets):
    spans, cur_l, cs, ce = [], None, -1, -1

    def flush():
        if cur_l is not None:
            s, e, l = clean_span(doc_text, cs, ce, cur_l)
            if e > s:
                spans.append([s, e, l])

    for t_id, (s_c, e_c) in zip(pred_ids, offsets):
        tag = ID2LABEL.get(int(t_id), "O")
        if tag.startswith("B-"):
            flush(); cur_l, cs, ce = tag[2:], s_c, e_c
        elif tag.startswith("I-") and cur_l == tag[2:]:
            ce = e_c
        elif tag.startswith("I-"):
            flush(); cur_l, cs, ce = tag[2:], s_c, e_c
        else:
            flush(); cur_l = None
    flush()
    seen, out = set(), []
    for s in spans:
        if tuple(s) not in seen:
            seen.add(tuple(s)); out.append(s)
    return out

NEG = -1e9
ALLOWED = np.ones((NUM_CLASSES, NUM_CLASSES), dtype=bool)
for cur in range(NUM_CLASSES):
    if ID2LABEL[cur].startswith("I-"):
        for prev in range(NUM_CLASSES):
            ALLOWED[prev, cur] = (ID2LABEL[prev] != "O") and (ID2LABEL[prev][2:] == ID2LABEL[cur][2:])
START_OK = np.array([not ID2LABEL[c].startswith("I-") for c in range(NUM_CLASSES)])

def make_trans(cont_bonus=0.0):
    t = np.where(ALLOWED, 0.0, NEG).astype(np.float64)
    for cur in range(NUM_CLASSES):
        if ID2LABEL[cur].startswith("I-"):
            for prev in range(NUM_CLASSES):
                if ALLOWED[prev, cur]:
                    t[prev, cur] += cont_bonus
    return t

def viterbi(logp, trans):
    T, C = logp.shape
    score = logp[0].copy()
    score[~START_OK] = NEG
    back = np.zeros((T, C), dtype=np.int64)
    for t in range(1, T):
        cand = score[:, None] + trans
        back[t] = cand.argmax(0)
        score = cand.max(0) + logp[t]
    path = [int(score.argmax())]
    for t in range(T - 1, 0, -1):
        path.append(int(back[t, path[-1]]))
    return path[::-1]

@torch.no_grad()
def doc_logp(text):
    model.eval()
    ids, offs = tokenizer.encode_with_exact_offsets(text)
    if not ids:
        return None, None
    acc = torch.zeros(len(ids), NUM_CLASSES)
    cnt = torch.zeros(len(ids), 1)
    for s in window_starts(len(ids)):
        e = min(len(ids), s + MAX_SEQ_LEN)
        chunk = ids[s:e]
        x = torch.tensor([chunk + [0] * (MAX_SEQ_LEN - len(chunk))], dtype=torch.long, device=DEVICE)
        with torch.amp.autocast("cuda", enabled=torch.cuda.is_available()):
            lp = F.log_softmax(model(x)[0].float(), dim=-1)[: len(chunk)].cpu()
        acc[s:e] += lp
        cnt[s:e] += 1
    return (acc / cnt).numpy(), offs

def decode_cached(cache, texts, o_bias=0.0, cont_bonus=0.0):
    trans, out = make_trans(cont_bonus), {}
    for d, (lp, offs) in cache.items():
        if lp is None:
            out[d] = []
            continue
        lp = lp.astype(np.float64).copy()
        lp[:, 0] += o_bias
        out[d] = decode_spans_exact(texts[d], viterbi(lp, trans), offs)
    return out

def strict_macro_f1(pred, gold, labels=LABEL_NAMES):
    f1s = []
    for lbl in labels:
        tp = fp = fn = 0
        for d in gold:
            g = {(a[0], a[1]) for a in gold[d] if a[2] == lbl}
            p = {(a[0], a[1]) for a in pred.get(d, []) if a[2] == lbl}
            tp += len(g & p); fp += len(p - g); fn += len(g - p)
        pr = tp / (tp + fp) if tp + fp else 0.0
        rc = tp / (tp + fn) if tp + fn else 0.0
        f1s.append(2 * pr * rc / (pr + rc) if pr + rc else 0.0)
    return sum(f1s) / len(f1s), dict(zip(labels, f1s))

In [11]:
counts = torch.zeros(NUM_CLASSES)
for s in train_ds.samples:
    l = s["labels"]
    l = l[l != IGNORE_INDEX]
    counts += torch.bincount(l, minlength=NUM_CLASSES).float()
w = (counts.sum() / counts.clamp(min=1)) ** 0.5
w = (w / w[0]).clamp(max=20.0)
print("O fraction:", round((counts[0] / counts.sum()).item(), 4))
criterion = nn.CrossEntropyLoss(weight=w.to(DEVICE), ignore_index=IGNORE_INDEX)

head_params = list(model.classifier.parameters())
head_ids = {id(p) for p in head_params}
backbone_params = [p for p in model.parameters() if id(p) not in head_ids]
optimizer = torch.optim.AdamW([
    {"params": backbone_params, "lr": BACKBONE_LR},
    {"params": head_params, "lr": HEAD_LR},
], weight_decay=0.01)
scaler = torch.amp.GradScaler("cuda", enabled=torch.cuda.is_available())

steps_per_epoch = math.ceil(len(train_loader) / GRAD_ACCUM_STEPS)
total_steps = steps_per_epoch * EPOCHS
warmup = max(1, int(0.05 * total_steps))

def lr_lambda(s):
    if s < warmup:
        return (s + 1) / warmup
    prog = (s - warmup) / max(1, total_steps - warmup)
    return 0.03 + 0.97 * 0.5 * (1 + math.cos(math.pi * prog))

scheduler = torch.optim.lr_scheduler.LambdaLR(optimizer, lr_lambda)

print(f"Training {EPOCHS} epochs | USE_VAL={USE_VAL}")
best_f1, best_state, best_epoch = -1.0, None, 0

for epoch in range(EPOCHS):
    model.train()
    running, n_log = 0.0, 0
    optimizer.zero_grad()
    for step, batch in enumerate(train_loader):
        input_ids = batch["input_ids"].to(DEVICE)
        labels = batch["labels"].to(DEVICE)
        with torch.amp.autocast("cuda", enabled=torch.cuda.is_available()):
            logits = model(input_ids)
        loss = criterion(logits.float().view(-1, NUM_CLASSES), labels.view(-1))
        scaler.scale(loss / GRAD_ACCUM_STEPS).backward()
        running += loss.item()
        n_log += 1

        if (step + 1) % GRAD_ACCUM_STEPS == 0 or (step + 1) == len(train_loader):
            scaler.unscale_(optimizer)
            torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
            scaler.step(optimizer)
            scaler.update()
            optimizer.zero_grad()
            scheduler.step()

        if (step + 1) % 100 == 0:
            print(f"Epoch [{epoch + 1}/{EPOCHS}] | Step [{step + 1}/{len(train_loader)}] | Loss: {running / n_log:.4f}")
            running, n_log = 0.0, 0

    if USE_VAL:
        vc = {d: doc_logp(t) for d, t in val_texts.items()}
        f1, per = strict_macro_f1(decode_cached(vc, val_texts, 1.5, 2.0), val_golds)
        print(f"==> Epoch {epoch + 1} | VAL strict macro-F1: {f1:.4f}")
        print({k: round(v, 3) for k, v in per.items()})
        if f1 > best_f1:
            best_f1, best_epoch = f1, epoch + 1
            best_state = {k: v.detach().cpu().clone() for k, v in model.state_dict().items()}

if USE_VAL and best_state is not None:
    model.load_state_dict(best_state)
    print(f"Restored best epoch {best_epoch} (val F1 {best_f1:.4f})")
print("Training finished")

O fraction: 0.9023
Training 6 epochs | USE_VAL=True
Epoch [1/6] | Step [100/163] | Loss: 2.2287
==> Epoch 1 | VAL strict macro-F1: 0.0391
{'APP': 0.022, 'RESP': 0.0, 'A.COUNSEL': 0.0, 'R.COUNSEL': 0.0, 'JUDGE': 0.031, 'WIT': 0.0, 'AUTH': 0.0, 'COURT': 0.0, 'STAT': 0.0, 'PREC': 0.0, 'DATE': 0.345, 'CASENO': 0.071}
Epoch [2/6] | Step [100/163] | Loss: 1.5691
==> Epoch 2 | VAL strict macro-F1: 0.1210
{'APP': 0.0, 'RESP': 0.0, 'A.COUNSEL': 0.0, 'R.COUNSEL': 0.0, 'JUDGE': 0.061, 'WIT': 0.0, 'AUTH': 0.223, 'COURT': 0.605, 'STAT': 0.154, 'PREC': 0.009, 'DATE': 0.34, 'CASENO': 0.061}
Epoch [3/6] | Step [100/163] | Loss: 1.2813
==> Epoch 3 | VAL strict macro-F1: 0.1299
{'APP': 0.0, 'RESP': 0.0, 'A.COUNSEL': 0.0, 'R.COUNSEL': 0.0, 'JUDGE': 0.036, 'WIT': 0.0, 'AUTH': 0.217, 'COURT': 0.538, 'STAT': 0.226, 'PREC': 0.038, 'DATE': 0.391, 'CASENO': 0.113}
Epoch [4/6] | Step [100/163] | Loss: 1.0924
==> Epoch 4 | VAL strict macro-F1: 0.1534
{'APP': 0.012, 'RESP': 0.011, 'A.COUNSEL': 0.0, 'R.COUNSEL': 0

In [18]:
val_cache = {d: doc_logp(t) for d, t in val_texts.items()}

# stage 1: O bias x continuation bonus
best = (-1.0, None)
for ob in [0.0, 1.0, 1.5, 2.0, 3.0]:
    for cb in [0.0, 0.5, 1.0, 2.0]:
        f, _ = strict_macro_f1(decode_cached(val_cache, val_texts, ob, cb), val_golds)
        print(f"o_bias={ob} cont_bonus={cb} -> {f:.4f}")
        if f > best[0]:
            best = (f, (ob, cb))
BEST_O_BIAS, BEST_CONT = best[1]
print("best (o_bias, cont_bonus):", best)

# stage 2: merge gap on top of the best setting
BEST_MAX_GAP, best_g = 0, best[0]
for g in [1, 2, 3]:
    f, _ = strict_macro_f1(decode_cached(val_cache, val_texts, BEST_O_BIAS, BEST_CONT, g), val_golds)
    print(f"max_gap={g} -> {f:.4f}")
    if f > best_g:
        best_g, BEST_MAX_GAP = f, g
print(f"\nFINAL SETTINGS: o_bias={BEST_O_BIAS} cont_bonus={BEST_CONT} max_gap={BEST_MAX_GAP} | val F1={best_g:.4f}")

final_val = decode_cached(val_cache, val_texts, BEST_O_BIAS, BEST_CONT, BEST_MAX_GAP)
final_val = {d: [[int(a[0]), int(a[1]), str(a[2])] for a in sp] for d, sp in final_val.items()}
evaluate_predictions_strictly(val_golds, final_val)

o_bias=0.0 cont_bonus=0.0 -> 0.0967
o_bias=0.0 cont_bonus=0.5 -> 0.1113
o_bias=0.0 cont_bonus=1.0 -> 0.1255
o_bias=0.0 cont_bonus=2.0 -> 0.1443
o_bias=1.0 cont_bonus=0.0 -> 0.0823
o_bias=1.0 cont_bonus=0.5 -> 0.1058
o_bias=1.0 cont_bonus=1.0 -> 0.1260
o_bias=1.0 cont_bonus=2.0 -> 0.1515
o_bias=1.5 cont_bonus=0.0 -> 0.0662
o_bias=1.5 cont_bonus=0.5 -> 0.0932
o_bias=1.5 cont_bonus=1.0 -> 0.1203
o_bias=1.5 cont_bonus=2.0 -> 0.1534
o_bias=2.0 cont_bonus=0.0 -> 0.0461
o_bias=2.0 cont_bonus=0.5 -> 0.0734
o_bias=2.0 cont_bonus=1.0 -> 0.1080
o_bias=2.0 cont_bonus=2.0 -> 0.1488
o_bias=3.0 cont_bonus=0.0 -> 0.0103
o_bias=3.0 cont_bonus=0.5 -> 0.0264
o_bias=3.0 cont_bonus=1.0 -> 0.0573
o_bias=3.0 cont_bonus=2.0 -> 0.1237
best (o_bias, cont_bonus): (0.15340285223387284, (1.5, 2.0))


TypeError: decode_cached() takes from 2 to 4 positional arguments but 5 were given

In [19]:
print(f"Generating predictions for {len(test_texts)} test documents...")
model.eval()
if "doc_logp" in globals() and "decode_cached" in globals():
  o_bias = globals().get("BEST_O_BIAS", 1.5)
  cont = globals().get("BEST_CONT", 2.0)
  test_cached = {d: doc_logp(t) for d, t in test_texts.items()}
  test_predictions = decode_cached(test_cached, test_texts, o_bias, cont)
else:
  # Standard exact argmax sliding window inference fallback:
  test_predictions = {}
  with torch.no_grad():
    for doc_id, text in test_texts.items():
      token_ids, offsets = tokenizer.encode_with_exact_offsets(text)
      if not token_ids:
        test_predictions[doc_id] = []
        continue
      doc_spans = []
      for start_idx in range(0, max(1, len(token_ids)), MAX_SEQ_LEN - STRIDE):
        end_idx = min(len(token_ids), start_idx + MAX_SEQ_LEN)
        chunk_ids = token_ids[start_idx:end_idx]
        chunk_offsets = offsets[start_idx:end_idx]
        input_tensor = torch.tensor(
            [chunk_ids], dtype=torch.long, device=DEVICE
        )
        with torch.amp.autocast("cuda", enabled=torch.cuda.is_available()):
          logits = model(input_tensor)
        preds = torch.argmax(logits[0], dim=-1).cpu().tolist()
        spans = decode_spans_exact(text, preds, chunk_offsets)
        doc_spans.extend(spans)
      final_spans = []
      for sp in doc_spans:
        if not any(
            sp[0] == s[0] and sp[1] == s[1] and sp[2] == s[2]
            for s in final_spans
        ):
          final_spans.append([int(sp[0]), int(sp[1]), str(sp[2])])
      test_predictions[doc_id] = final_spans
print(f"✓ Created test_predictions for {len(test_predictions)} documents!")

Generating predictions for 35 test documents...
✓ Created test_predictions for 35 documents!


In [20]:
# BEST_O_BIAS, BEST_CONT, BEST_MAX_GAP come from Cell 7 (or set them by hand, e.g. 2.0, 0.5, 0)

output_json = "/kaggle/working/prediction.json"
output_zip = "/kaggle/working/submission.zip"
with open(output_json, "w", encoding="utf-8") as f:
    f.write(json.dumps(test_predictions))
    f.flush()
    os.fsync(f.fileno())
if os.path.exists(output_zip):
    os.remove(output_zip)
with zipfile.ZipFile(output_zip, "w", compression=zipfile.ZIP_DEFLATED) as zf:
    zf.write(output_json, arcname="prediction.json")
print(f"\n✓ Submission zip created: {output_zip} ({os.path.getsize(output_zip):,} bytes)")

with zipfile.ZipFile(output_zip, "r") as zf:
    verified = json.loads(zf.read("prediction.json").decode("utf-8"))
    first_k = list(verified.keys())[0]
    print(f"✓ ZIP VERIFIED: Found {len(verified)} documents.")
    print(f"Sample prediction for doc {first_k}: {verified[first_k][:3]}")


✓ Submission zip created: /kaggle/working/submission.zip (45,334 bytes)
✓ ZIP VERIFIED: Found 35 documents.
Sample prediction for doc 1128835: [[0, 69, 'RESP'], [70, 98, 'CASENO'], [99, 142, 'RESP']]
